# Preprocesamiento

1. Detectar gaps grandes y marcar límites de sesión
2. Remuestrear de segundos a minutos con agregación correcta por tipo de señal
3. Filtrar régimen operacional (Motor_current > umbral)
4. Identificar y numerar sesiones operacionales
5. Identificar regimen 0, 1 y 2
6. Guardar en data/processed/operacional.parquet

In [56]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

sns.set_style('whitegrid')

ANALOG_COLS  = ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs',
                'Oil_temperature', 'Motor_current', 'Caudal_impulses']
BINARY_COLS  = ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS',
                'Pressure_switch', 'Oil_level']

In [57]:
datos = pd.read_csv('../data/raw/MetroPT3(AirCompressor).csv',
                parse_dates=['timestamp'])
datos.drop(columns=['Unnamed: 0'], inplace=True)
df = datos.sort_values('timestamp').reset_index(drop = True)
df.head()

,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


# 1. Detectar gaps grandes y marcar límites de sesión

#### Verificar fracuencia real (aparentemente son 10 segundos)

In [58]:
diferencias = df['timestamp'].diff().dt.total_seconds()

print('Frecuencia predominante (segundos):')
print(diferencias.value_counts().head(10))

print("\nGaps mayores a 30 segundos:")
print((diferencias > 30).sum())

print(f'\nGap maximo {diferencias.max():.0f} segundos')
print(f'Gap máximo {diferencias.max()/60:.0f} minutos')
print(f'Gap máximo {diferencias.max()/3600:.0f} horas')

Frecuencia predominante (segundos):
timestamp
10.0    1337521
9.0      128277
12.0      38321
13.0       7988
11.0       4471
21.0         10
19.0          5
22.0          4
20.0          3
17.0          3
Name: count, dtype: int64

Gaps mayores a 30 segundos:
331

Gap maximo 172918 segundos
Gap máximo 2882 minutos
Gap máximo 48 horas


#### Detectar gaps grandes antes de remeustrear

Cuando los gaps son grandes, significa que no fue un error de registro y si fue un paro de operación real y por lo tanto, el final de una sesión de uso.

Con lo anterior se puede ver que hay variaciónes de hasta 13 segundos que se pueden considerar como normales. (7988)

In [ ]:
GAP_UMBRAL = 60 # segundos o 1 minuto
# La variación normal más grande es de 13 segundos.
# 1 minuto es como un intervalo de seguridad para esos 13 segundos

df['diff_segundos'] = df['timestamp'].diff().dt.total_seconds()
df['nuevo_segmento'] = (df['diff_segundos'] > GAP_UMBRAL)
df['id_segmento'] = df['nuevo_segmento'].cumsum()

print(f"Total de segmentos: {df['id_segmento'].nunique()})")
df.head(10)

Total de segmentos: 332


,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses,diff_segundos,nuevo_segmento,id_segmento
0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,NaN,False,0
1,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,10.0,False,0
2,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,9.0,False,0
3,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,10.0,False,0
4,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,10.0,False,0
5,2020-02-01 00:00:49,-0.012,9.306,9.290,-0.024,9.308,53.500,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,10.0,False,0
6,2020-02-01 00:00:59,-0.012,9.296,9.280,-0.024,9.298,53.375,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,10.0,False,0
7,2020-02-01 00:01:09,-0.014,9.286,9.270,-0.024,9.286,53.550,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,10.0,False,0
8,2020-02-01 00:01:19,-0.012,9.276,9.258,-0.022,9.276,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,10.0,False,0
9,2020-02-01 00:01:29,-0.012,9.264,9.248,-0.022,9.264,53.375,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,10.0,False,0


In [73]:
tamaños = df.groupby('id_segmento').size()*10/60 # en minutos

print('Tamaño de los segmentos (minutos)')
print(tamaños.describe())

print('\nSegmentos menores a 10 minutos')
print(f'{(tamaños < 10).sum()}')

print('\nSegmentos mayores a 60 minutos')
print(f'{(tamaños > 60).sum()}')

print('\nSegmentos mayroes a 6 hroas')
print(f'{(tamaños > 360).sum()}')

Tamaño de los segmentos (minutos)
count     332.000000
mean      761.520080
std       957.510374
min         4.833333
25%        25.000000
50%       389.833333
75%      1205.000000
max      6099.833333
dtype: float64

Segmentos menores a 10 minutos
41

Segmentos mayores a 60 minutos
213

Segmentos mayroes a 6 hroas
170


De los 332 segmentos:

| Métrica | Valor | Equivalencia |
|---------|-------|---------------|
| **Media** | 761 min | ≈ 12.7 h |
| **Mínimo** | 4.8 min | ≈ 5 min |
| **Máximo** | 6099 min | ≈ 101 h (4.2 días) |



- **25%** de los segmentos duraron **≤ 25 minutos**
- **50%** (mediana) duraron **≤ 389.8 minutos** (≈ 6.5 h)
- **75%** duraron **≤ 1205 minutos** (≈ 20 h)



| Condición | N° segmentos | Destino |
|-----------|--------------|---------|
| **< 10 minutos** | 41 | Descartar porque son muy poquitos |
| **> 60 minutos** | 213 | Dicen muy poco |
| **> 6 horas (360 min)** | 170 | **Núcleo para entrenar** |

